<a href="https://colab.research.google.com/github/bmf87/pydata_copilot/blob/main/notebook/create_pthon_coding_prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS 552 - Generative AI
* Capstone Project Title: PyData Copilot
* Objective: Create JSON List EDA Prompt File
* Author: Brett Favro

## Notebook Dependencies

In [ ]:
import itertools, json, random
from datasets import load_dataset
from google.colab import drive

In [ ]:
# Mount Google Drive
drive.mount('/content/drive/')
data_dir = '/content/drive/MyDrive/Colab Notebooks/DS552/project/data'

Mounted at /content/drive/


## List Configuration
* Static lists to drive JSONL creation logic

In [ ]:
# 10 Domains: Sythetic Business Domains
DOMAINS = [
    "sales",
    "marketing",
    "finance",
    "healthcare",
    "web_traffic",
    "ecommerce",
    "operations",
    "education",
    "HR",
    "life_sciences",
]
# 10 Tasks Types: Data Analysis tasks
TASK_TYPES = [
    "groupby",
    "groupby_plot",
    "merge_groupby",
    "pivot_plot",
    "timeseries",
    "timeseries_plot",
    "distribution_plot",
    "correlation_plot",
    "seaborn_facet",
    "cleaning",
]

# grouped by task_type
PROMPT_TEMPLATES = {
    "groupby": [
        "Using pandas, group the {domain} dataset by a key category and compute total and average values for one or more numeric metrics.",
        "Using pandas, group the {domain} data by a categorical key and calculate count, mean, and sum for one or more numeric columns.",
        "With pandas, aggregate the {domain} dataset by a category and compute multiple summary statistics (e.g., mean, median, max) for selected numeric fields.",
        "Using pandas groupby on a key column in the {domain} data, produce a summary table with at least two aggregated metrics per group."
    ],
    "groupby_plot": [
        "Using pandas and matplotlib, group the {domain} dataset by a category and plot a bar chart showing the total of a numeric metric per group.",
        "Aggregate the {domain} data by a categorical feature with pandas, then create a bar plot comparing the average value of a numeric column across groups.",
        "With pandas and matplotlib, compute group-level statistics for the {domain} data and visualize them as a labeled bar chart."
    ],
    "merge_groupby": [
        "Using pandas, join two related {domain} DataFrames on a shared key, then group the merged result and compute aggregated metrics (such as counts and averages).",
        "Merge two {domain} tables on a primary/foreign key using pandas, then perform a groupby on a categorical column and summarize key numeric fields.",
        "Combine two {domain} datasets with a pandas merge on a key column, then group the combined data to calculate summary statistics for each group."
    ],
    "pivot_plot": [
        "Build a pandas pivot table for the {domain} data that shows a numeric metric across two categorical dimensions, then plot it as a seaborn heatmap.",
        "Using pandas, pivot the {domain} dataset to form a matrix of values by row and column categories, and visualize the result using a seaborn heatmap with colorbar.",
        "Create a pivot table from the {domain} data summarizing a numeric column by two categorical keys, then display it as a heatmap with seaborn."
    ],
    "timeseries": [
        "Using pandas, resample the time-indexed {domain} data to a coarser frequency (such as daily or monthly) and compute aggregates like sum and mean.",
        "For the {domain} time series dataset, resample to a specified interval using pandas and calculate aggregated statistics for a numeric column.",
        "With pandas, take the {domain} dataset indexed by timestamps and resample it to a regular frequency, computing summary metrics for each period."
    ],
    "timeseries_plot": [
        "Using pandas and matplotlib, plot a line chart of a time series from the {domain} data and overlay a rolling mean for smoothing.",
        "From the {domain} time series dataset, create a matplotlib plot showing the original values and a moving average on the same axes.",
        "Plot a time series from the {domain} data with pandas/matplotlib, including both the raw series and a rolling-window average to highlight trends."
    ],
    "distribution_plot": [
        "Visualize the distribution of a numeric feature in the {domain} dataset using a histogram, adding labels and a vertical line for a key statistic (e.g., mean or median).",
        "Using pandas and matplotlib or seaborn, plot a histogram (or density plot) of a numeric column from the {domain} data and annotate important summary values.",
        "Create a histogram for a numeric variable in the {domain} dataset, adjust binning appropriately, and include a title and axis labels."
    ],
    "correlation_plot": [
        "Compute a correlation matrix for selected numeric columns in the {domain} data and plot it as a seaborn heatmap with annotated correlation values.",
        "Using pandas, calculate correlations between key numeric fields in the {domain} dataset, then visualize them with a formatted seaborn heatmap.",
        "Derive a correlation matrix from the {domain} data for important numeric features and display it using a seaborn heatmap with a colorbar."
    ],
    "seaborn_facet": [
        "Create a seaborn FacetGrid for the {domain} dataset that shows the relationship between a numeric variable and a category across multiple subplots.",
        "Using seaborn's FacetGrid, visualize a metric from the {domain} data faceted by one or two categorical variables (e.g., row/column facets).",
        "Build a FacetGrid from the {domain} dataset to compare the distribution or trend of a numeric feature across different category levels."
    ],
    "cleaning":[
        "Clean the {domain} dataset with pandas by handling missing values, converting columns to appropriate dtypes, removing duplicates, and summarizing the changes.",
        "Using pandas, perform basic data cleaning on the {domain} DataFrame: fix dtypes, handle nulls, drop obvious duplicates, and print a brief before/after overview.",
        "Apply common pandas data-cleaning steps to the {domain} dataset (missing value handling, type correction, duplicate removal) and display a concise summary."
    ]
}


PERSONAS = [
    "",
    "You are a senior data analyst. ",
    "I need some quick help with this. ",
    "As part of a data pipeline, ",
    "For our upcoming presentation, "
]

CONSTRAINTS = [
    "",
    " Please ensure the code is well-commented.",
    " Store the result in a variable named 'final_df'.",
    " Make sure to handle any missing values properly.",
    " Reset the index of the resulting DataFrame.",
    " Focus on writing performant and vectorized code.",
    " Avoid chained indexing and explicitly use .loc.",
]

SYNONYMS = {
    "dataset": ["dataset", "data", "DataFrame", "records", "table"],
    "compute": ["compute", "calculate", "find", "determine"],
    "visualize": ["visualize", "plot", "chart", "graph"],
    "a category": ["a category", "a categorical feature", "a grouping variable", "a distinct category"],
    "a numeric metric": ["a numeric metric", "a continuous value", "a numeric column", "a quantitative field"]
}

# grouped by task_type
STYLE_HINTS = {
    "groupby": [
        "Prefer vectorized pandas operations over explicit Python for-loops.",
        "Use groupby + agg with named aggregations instead of multiple groupby calls.",
        "Flatten multi-index columns after groupby for readability.",
        "Use value_counts with normalize=True when computing simple distributions.",
        "Use meaningful column and variable names that reflect the business meaning.",
        "Avoid chained indexing; use .loc and .iloc explicitly for selection.",
        "Show intermediate groupby results in variables rather than deeply nested expressions.",
        "Document non-obvious groupby logic with a short comment above the code.",
    ],
    "groupby_plot": [
        "Prefer vectorized pandas operations over explicit Python for-loops.",
        "Use groupby + agg with named aggregations before plotting.",
        "Always label axes and add a descriptive title for each plot.",
        "Rotate x-axis tick labels when there are many categories to avoid overlap.",
        "Use tight_layout() or adjust subplot spacing so labels are not clipped.",
        "Choose bar plots for aggregated categories rather than line plots.",
        "Use consistent color palettes across related groupby plots.",
        "Sort categories by a meaningful metric before plotting for readability.",
    ],
    "merge_groupby": [
        "Use explicit merge(...) with how= specified rather than relying on defaults.",
        "Check and align key column names before merging instead of renaming implicitly.",
        "After merge, use groupby + agg with named aggregations for clarity.",
        "Drop unnecessary columns after merge to keep the result tidy.",
        "Avoid chained indexing; use .loc and .iloc explicitly for selection.",
        "Use meaningful column and variable names that reflect the business meaning.",
        "Show intermediate merged and grouped results in variables instead of nesting operations.",
        "Guard against duplicate keys when merging if they would distort aggregates.",
    ],
    "pivot_plot": [
        "Use pivot_table for multi-dimensional summaries instead of manual groupby reshaping.",
        "Flatten multi-index columns after pivot_table for readability.",
        "Use seaborn heatmaps with clearly labeled axes and a colorbar.",
        "Use tight_layout() so labels and colorbars are not clipped.",
        "Choose an appropriate colormap for the metric (e.g., diverging for centered data).",
        "Annotate cells or use fmt options when exact values aid interpretation.",
        "Mask or hide cells with insufficient data to avoid misleading visuals.",
        "Sort rows and columns in the pivot table by meaningful business order.",
    ],
    "timeseries": [
        "Convert date-like columns to datetime before resampling.",
        "Set the datetime column as an index when doing time series operations.",
        "Use resample with explicit frequency strings (e.g., 'D', 'M') and clear aggregations.",
        "Handle missing values explicitly before computing rolling or resampled metrics.",
        "Use meaningful names for resampled columns (e.g., 'monthly_total_sales').",
        "Guard against timezone issues when combining multiple time series.",
        "Avoid chained indexing; use .loc with date slices for clarity.",
        "Document non-obvious resampling choices (e.g., end vs start of period).",
    ],
    "timeseries_plot": [
        "Convert date-like columns to datetime and set them as an index before plotting.",
        "Use line plots for time series rather than bar plots by default.",
        "Overlay rolling averages with a clear legend entry.",
        "Label axes and add a descriptive title that includes the time granularity.",
        "Rotate date tick labels to avoid overlap.",
        "Use tight_layout() to prevent clipping of date labels.",
        "Use a consistent date format on the x-axis for readability.",
        "Annotate important events or thresholds directly on the time series plot.",
    ],
    "distribution_plot": [
        "Use histplot or hist with an appropriate number of bins for the sample size.",
        "When plotting distributions, combine histograms with KDE only when it aids interpretation.",
        "Use log scales on axes when the data is highly skewed and it improves readability.",
        "Label axes and include units where applicable.",
        "Annotate important thresholds or reference lines in the plot, such as targets or percentiles.",
        "Use facet plots to compare distributions across key segments (e.g., region or cohort).",
        "Avoid over-plotting by adjusting transparency (alpha) when overlaying distributions.",
        "Use consistent color palettes across related distribution plots.",
    ],
    "correlation_plot": [
        "Select a focused set of numeric features before computing correlations.",
        "Use correlation heatmaps and mask the upper triangle for clarity.",
        "Annotate correlation coefficients when the matrix is small enough.",
        "Use a diverging colormap centered at zero for correlation heatmaps.",
        "Label axes clearly with feature names and rotate labels if needed.",
        "Sort features by domain-relevant groups to make patterns more interpretable.",
        "Drop constant or near-constant columns before computing correlations.",
        "Document any feature engineering that materially affects correlations.",
    ],
    "seaborn_facet": [
        "Use facet plots to compare distributions or trends across key segments.",
        "Keep each facet uncluttered by limiting the number of plotted series per subplot.",
        "Share axes across facets when comparisons of scale are important.",
        "Provide clear titles or facet labels describing each subset.",
        "Use consistent color palettes across facets to maintain comparability.",
        "Rotate x-axis tick labels if facets contain many categories.",
        "Use tight_layout() or facet-specific spacing parameters to avoid overlapping labels.",
        "Limit the number of facets or paginate when there are too many categories.",
    ],
    "cleaning": [
        "Handle missing values explicitly before computing aggregates or plots.",
        "Convert date-like columns to datetime and categorical fields to category dtype when appropriate.",
        "Remove exact duplicate rows using drop_duplicates and document the impact.",
        "Avoid modifying DataFrames in place when it reduces clarity; favor explicit assignments.",
        "Show a concise before/after summary of row counts and missing values.",
        "Use .astype with care and validate conversions instead of relying on implicit casts.",
        "Standardize string columns (e.g., trim whitespace, consistent casing) before analysis.",
        "Document non-obvious cleaning rules, such as outlier handling or custom filters.",
    ]
}

## Functions - to create DPO Prompts
* Create prompt file based on domain + task_type tuples = 10 x 10 x ~3 prompt templates per task_type = 300
* Combinatorial Prompt Construction: added lists for Persona and Constraints
  * Net Variability: 300 × 5 Personas × 7 Constraints × (roughly 3-4 synonym swaps per string) = Over 30,000 unique text prompts

In [ ]:
# def make_prompt(domain: str, task_type: str) -> str:
#     templates = PROMPT_TEMPLATES.get(task_type)
#     if templates:
#         return random.choice(templates).format(domain=domain)
#     # fallback / baseline prompt
#     return f"Perform an exploratory data analysis task on the {domain} dataset using pandas."

def make_prompt(domain: str, task_type: str) -> str:
    templates = PROMPT_TEMPLATES.get(task_type)
    if templates:
        base_prompt = random.choice(templates).format(domain=domain)
    else:
        # fallback / baseline prompt
        base_prompt = f"Perform an exploratory data analysis task on the {domain} dataset using pandas."

    # Synonym Replacement
    for word, alternatives in SYNONYMS.items():
        if word in base_prompt:
            base_prompt = base_prompt.replace(word, random.choice(alternatives))

    # Add Persona Wrapping and Constraints for diversity
    persona = random.choice(PERSONAS)
    constraint = random.choice(CONSTRAINTS)

    # Capitalization fix
    final_prompt = f"{persona}{base_prompt}{constraint}".strip()
    return final_prompt[0].upper() + final_prompt[1:]


def attach_style_hints(task_type: str, k: int = 3) -> list[str]:
    hints = STYLE_HINTS.get(task_type, [])
    if not hints:
        return []
    k = min(k, len(hints))
    return random.sample(hints, k)

def make_data_hints(domain: str, task_type: str) -> str:
    """Lightweight data_hints text by domain."""
    if domain == "life_sciences":
        return "Columns may include patient_id, treatment_group, biomarker_value, visit_date."
    if domain == "sales":
        return "Columns may include order_id, customer_id, region, order_date, revenue."
    if domain == "marketing":
        return "Columns may include campaign_id, channel, impressions, clicks, conversions."
    if domain == "finance":
        return "Columns may include date, instrument, price, volume, and daily_return."
    if domain == "healthcare":
        return "Columns may include patient_id, visit_id, diagnosis, department, admission_date."
    if domain == "web_traffic":
        return "Columns may include date, page_views, sessions, device_type, country."
    if domain == "ecommerce":
        return "Columns may include order_id, customer_id, product_id, category, order_value."
    if domain == "operations":
        return "Columns may include machine_id, line_id, runtime_hours, output_units, defects."
    if domain == "education":
        return "Columns may include student_id, class_id, subject, score, exam_date."
    if domain == "HR":
        return "Columns may include employee_id, department, job_level, salary, tenure_years."
    # generic fallback
    return "Includes a mix of categorical and numeric columns relevant to the domain."

def pick_libraries(task_type: str):
    libs = ["pandas"]
    if any(k in task_type for k in ["plot", "timeseries", "distribution"]):
        libs.append("matplotlib")
    if any(k in task_type for k in ["seaborn", "correlation_plot", "facet", "pivot_plot"]):
        libs.append("seaborn")
    return sorted(set(libs))

def default_output_focus(task_type: str) -> str:
    if any(k in task_type for k in ["facet"]):
        return "multi_plot"
    if any(k in task_type for k in ["plot", "distribution", "timeseries_plot", "correlation_plot", "pivot_plot"]):
        return "single_plot"
    return "table"

def default_plot_required(task_type: str) -> bool:
    return any(k in task_type for k in ["plot", "facet", "distribution", "timeseries_plot", "correlation_plot", "pivot_plot"])

def random_difficulty() -> str:
    return random.choices(
        population=["easy", "medium", "hard"],
        weights=[0.4, 0.4, 0.2],
        k=1,
    )[0]


## Run: Create Expanded JSON
* Create new JSONL Prompt FIle
* Including the following:
  * Data hints
  * Expected libraries
  * **Style hints**: critical to vary candidate output for DPO selection

In [ ]:
# JSONL Prompt file
output_file = data_dir + "/code_prompts_10000.jsonl"

# Create synthethic prompts until >= target_total
#rows = base_rows[:]  # start with the original 100
rows = []
max_id = 0
target_total = 10000
current_id = max_id + 1

# Cartesian product domains x task types (10x10)
domain_task_pairs = list(itertools.product(DOMAINS, TASK_TYPES))

# Shuffle: interleave domain/task_type pairs
random.shuffle(domain_task_pairs)

# Generate multiple prompts per (domain, task_type) until target_total reached
while len(rows) < target_total:
    for domain, task_type in domain_task_pairs:
        if len(rows) >= target_total:
            break

        prompt_text = make_prompt(domain, task_type)
        data_hints = make_data_hints(domain, task_type)
        libs = pick_libraries(task_type)

        new_row = {
            "id": current_id,
            "domain": domain,
            "task_type": task_type,
            "difficulty": random_difficulty(),
            "prompt": prompt_text,
            "data_hints": data_hints,
            "expected_libraries": libs,
            "plot_required": default_plot_required(task_type),
            "output_focus": default_output_focus(task_type),
            "notes": "",
            "style_hints": attach_style_hints(task_type, k=5),
        }
        rows.append(new_row)
        current_id += 1

with open(output_file, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Wrote {len(rows)} rows to {output_file}")

Wrote 10000 rows to /content/drive/MyDrive/Colab Notebooks/DS552/project/data/code_prompts_10000.jsonl
